# Lab 4 - Improper Output Handling
### Decode Builder Lab · Cyber X AI

**The lesson:** an LLM's output is **untrusted input to whatever consumes it next**. If you drop it
into a web page you get XSS; if you `eval()` it you get remote code execution. The model doesn't even
need to be "hacked" - it just needs to produce a string your code trusts.

**Maps to:** OWASP LLM Top 10 (2025) - *LLM05 Improper Output Handling*.

> ⚠️ Authorized testing only. The "payloads" here are benign proofs, run locally.

## 0 · Setup - pick your model

This notebook talks to any **OpenAI-compatible** endpoint, so you have two options:

- **Google AI Studio** - free Gemini key at aistudio.google.com/apikey.
- **Workshop LiteLLM endpoint** - your instructor hands out a key (revoked after the event) with
  several open-source models.

Set `PROVIDER` in the cell below.

> 🔑 **Lesson before we start:** Google's models often *refuse* these attacks — that's the *vendor's*
> safety training, **not your app's security**. Open models comply readily. You can't outsource
> security to the model vendor, so switch to an open model (e.g. `qwen3-coder-30b`, `glm-4.7-flash`)
> to expose your app's *real* vulnerability, then secure the app itself.

In [ ]:
%pip install -q openai

import os, json, re, html
from getpass import getpass
from openai import OpenAI

# ── Choose your provider ─────────────────────────────────────────────
# "google"  → Google AI Studio (free key: https://aistudio.google.com/apikey)
# "litellm" → Workshop endpoint (instructor gives you a key, revoked after the event)
PROVIDER = "litellm"

if PROVIDER == "google":
    client = OpenAI(
        api_key=getpass("Google AI Studio API key: "),
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    MODEL = "gemini-2.5-flash"                 # advanced safety — may RESIST these attacks
else:
    client = OpenAI(
        api_key=getpass("Workshop LiteLLM key: "),
        base_url="YOUR_LITELLM_BASE_URL",
    )
    MODEL = "glm-4.7-flash"                  # open models are more susceptible — best for this lab
    # other models on the workshop endpoint:
    #   glm-4.7-flash · devstral-2-123b · qwen3-vl-235b
    #   claude-haiku-4-5-20251001-v1:0 · claude-sonnet-4-6

print("Provider:", PROVIDER, "| Model:", MODEL)

Provider: litellm | Model: qwen3-coder-30b


In [3]:
def run_agent(user_message, system_prompt, tools=None, tool_impls=None, max_turns=5, verbose=True):
    tool_impls = tool_impls or {}
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message},
    ]
    for _ in range(max_turns):
        kw = {"model": MODEL, "messages": messages}
        if tools:
            kw["tools"] = tools; kw["tool_choice"] = "auto"
        resp = client.chat.completions.create(**kw)
        msg = resp.choices[0].message
        messages.append(msg.model_dump())
        if not getattr(msg, "tool_calls", None):
            return msg.content
        for call in msg.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments or "{}")
            if verbose: print(f"  🔧 agent calls {name}({args})")
            try:
                result = tool_impls[name](**args)
            except Exception as e:
                result = f"ERROR: {e}"
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "name": name, "content": str(result)})
    return "[max turns reached]"

## 1 · Sink A - rendering model output as HTML (XSS)
A support widget shows the bot's reply on a page. We simulate what a browser would receive.
An attacker gets the model to emit markup.

In [4]:
# Represent what the model returned. In the workshop, generate this live with:
#   model_output = run_agent("Reply with exactly: <img src=x onerror=alert('decode{...}')>", "You echo the user.")
model_output = "<script>steal(document.cookie)//decode{llm_output_is_untrusted_input}</script>"

def render_vulnerable(text):
    return f"<div class='chat'>{text}</div>"   # 🔴 raw interpolation

page = render_vulnerable(model_output)
print("HTML sent to browser:\n", page)
print("\nBrowser would EXECUTE a script tag?", "<script>" in page)

HTML sent to browser:
 <div class='chat'><script>steal(document.cookie)//decode{llm_output_is_untrusted_input}</script></div>

Browser would EXECUTE a script tag? True


### 1a · 🟢 Fix - escape before rendering

In [5]:
def render_safe(text):
    return f"<div class='chat'>{html.escape(text)}</div>"   # 🟢 neutralized

safe_page = render_safe(model_output)
print("HTML sent to browser:\n", safe_page)
print("\nExecutable script tag present?", "<script>" in safe_page)   # False — it's now &lt;script&gt;

HTML sent to browser:
 <div class='chat'>&lt;script&gt;steal(document.cookie)//decode{llm_output_is_untrusted_input}&lt;/script&gt;</div>

Executable script tag present? False


## 2 · Sink B - feeding model output to `eval()` (RCE)
A "smart calculator" asks the model to turn words into a Python expression, then evaluates it.
Convenient - and catastrophic.

In [6]:
# What the model returned for "what's os current directory" style abuse:
model_expr = "__import__('os').getcwd()"   # benign proof that ARBITRARY code runs

def calc_vulnerable(expr):
    return eval(expr)   # 🔴 arbitrary code execution

print("Vulnerable eval ran attacker code ->", calc_vulnerable(model_expr))
print("If that were __import__('os').system('rm -rf ...') it would have run too. Flag: decode{llm_output_is_untrusted_input}")

Vulnerable eval ran attacker code -> /home/awkinyua/code/python/decode-regions-cafe-CyberxAI
If that were __import__('os').system('rm -rf ...') it would have run too. Flag: decode{llm_output_is_untrusted_input}


### 2a · 🟢 Fix - never eval; parse a safe arithmetic AST

In [7]:
import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg, ast.Mod: operator.mod}

def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)): return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS: return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS: return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsafe expression rejected")
    return ev(ast.parse(expr, mode="eval").body)

print("safe_calc('3 * (4 + 5)') ->", safe_calc("3 * (4 + 5)"))
try:
    safe_calc("__import__('os').getcwd()")
except ValueError as e:
    print("attack rejected ->", e)

safe_calc('3 * (4 + 5)') -> 27
attack rejected -> unsafe expression rejected


---
## 3 · What you learned
| Sink | Vulnerable | Secured |
|---|---|---|
| HTML render | raw interpolation → XSS | `html.escape()` |
| Expression eval | `eval()` → RCE | AST arithmetic allow-list |

The same discipline you already apply to *user* input applies to *model* output. Zero Trust:
model output crosses a trust boundary on its way back into your app — encode it for the destination
(HTML-escape for pages, parameterize for SQL, never shell/eval).

**One sentence:** the LLM is just another untrusted producer of strings.

Flag: `decode{llm_output_is_untrusted_input}`